# 04 · Export Power BI

In [1]:
%pip install -q geopandas xlrd openpyxl scikit-learn requests pyarrow

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.13/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


In [2]:
ROOT = "/lakehouse/default"
SEUIL_POP_DIFFUSION = 200

In [3]:
# Parameters
ROOT = "data/lakehouse"


In [4]:
import json
from pathlib import Path
import pandas as pd
from shapely import wkt
from shapely.geometry import mapping

OUT = Path(ROOT) / "Files/hackaton/out"
read = lambda name: spark.read.table(name).toPandas()

gold, sugg, couts = read("gold_iris"), read("gold_suggestions"), read("gold_couts")
gold = gold[gold.pop_2021 >= SEUIL_POP_DIFFUSION]

In [5]:
gold.to_csv(OUT / "export_powerbi_iris.csv", index=False)
sugg.to_csv(OUT / "export_powerbi_suggestions.csv", index=False)
couts.to_csv(OUT / "export_powerbi_couts.csv", index=False)
props = ["ville", "code_iris", "nom_iris", "nom_commune", "pop_2021", "pop_2030_proj", "tx_annuel_15_21", "tx_proj_21_30", "evolution_predite"]
for v, g in gold.groupby("ville"):
    fc = {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": r[props].to_dict(), "geometry": mapping(wkt.loads(r.geometry_wkt))} for _, r in g.iterrows()]}
    (OUT / f"iris_{v}.geojson").write_text(json.dumps(fc, default=float))
sorted(p.name for p in OUT.iterdir())

['export_powerbi_couts.csv',
 'export_powerbi_iris.csv',
 'export_powerbi_suggestions.csv',
 'iris_Bordeaux.geojson',
 'iris_Grenoble.geojson',
 'iris_Lille.geojson',
 'iris_Rennes.geojson',
 'iris_Toulouse.geojson',
 'ml_metrics.json',
 'top_opportunites.csv']